# PhysX-Omni 论文精读 第五步：PhysX-Bench 评测设计与数据字段

论文地址：[https://arxiv.org/abs/2605.21572v1](https://arxiv.org/abs/2605.21572v1)  
代码地址：[https://github.com/physx-omni/PhysX-Omni](https://github.com/physx-omni/PhysX-Omni)  
PhysX-Bench 数据集：[https://huggingface.co/datasets/PhysX-Omni/PhysX-Bench](https://huggingface.co/datasets/PhysX-Omni/PhysX-Bench)  
本地代码：`C:\Users\robot\Documents\成长学习库\physx-omni-assets\code\PhysX-Omni`

## 0. 一句话理解 PhysX-Bench

PhysX-Bench 是给 simulation-ready physical 3D generation 用的无 GT 评测基准。它不要求每张 in-the-wild 条件图都有对应的真实 3D 资产，而是把生成结果转成可视证据，再用固定 VLM prompt 评价几何外观、尺度、affordance、运动学、材料和语义描述。

它解决的是传统 3D benchmark 的一个核心缺口：传统指标更擅长评价几何和外观，但很难评价一个生成资产是否有合理尺度、可交互区域、关节行为、材料力学行为，以及 part-level 语义是否落在正确部件上。

## 1. Bench 的核心设计

PhysX-Bench 的设计可以拆成四层：

| 层级 | 做什么 | 为什么重要 |
|---|---|---|
| 条件图层 | 收集真实图片和合成渲染图片作为输入条件 | 覆盖真实场景和 synthetic asset，两者共同测试泛化 |
| 生成结果层 | 每个方法对同一个 object_id 生成 3D/物理资产 | 保证不同方法面对相同条件图 |
| 证据资产层 | 把 3D 结果渲染成 views、mask、heatmap、motion video、material video | VLM 不直接读内部参数，而是看可视化物理行为 |
| 评分与校验层 | 固定 prompt 评分，聚合 object-level/dataset-level，并校验分母 | 防止缺样本、缺证据导致平均分虚高 |

论文里强调的评测原则是：用 rendered images 或 videos 降低 VLM 理解复杂 3D 和物理属性的难度，而不是直接把原始物理参数塞给 VLM 让它抽象判断。

## 2. 数据来源与规模

官方 Hugging Face 数据集当前文件树包含：

| 数据域 | 条件图目录 | 数量 | object_id 形态 |
|---|---|---:|---|
| in-the-wild | `demo_inthewild/` | 426 | 连续或半连续数字字符串，如 `0.png`、`202.png` |
| mobility | `demo_mobility/` | 388 | PartNet-Mobility 风格数字 ID，如 `100028.png` |
| verse | `demo_verse/` | 400 | 32 位 hash 风格 ID，如 `00d1cb5aa82745228a3b764c97f867de.png` |
| 合计 | - | 1214 | - |

同时有三个 description JSON：

| 文件 | 条目数 | 字段结构 |
|---|---:|---|
| `descript_inthewild.json` | 426 | `{object_id: reference_description}` |
| `descript_mobility.json` | 388 | `{object_id: reference_description}` |
| `descript_verse.json` | 400 | `{object_id: reference_description}` |

本地 repo 的 `benchmark/assets/description/descript_*.json` 与 Hugging Face 当前版本在 key 和 value 上一致。README 很短，只声明 license 和 “Image source of PhysX-Bench”，真正的数据结构主要由文件树和 benchmark 代码约定。

## 3. 原始数据到本地评测布局

HF 原始数据：

```text
demo_<dataset>/<object_id>.png
descript_<dataset>.json
```

代码期望转换后的 condition image 布局：

```text
benchmark/benchmark_assets/condition_images/<dataset>/<object_id>/first_frame.png
```

转换脚本：

```bash
python3 benchmark/code/assets/prepare_demo_condition_images.py \
  --input-dir physx_result/demo_mobility \
  --dataset mobility \
  --output-root benchmark/benchmark_assets/condition_images \
  --symlink --skip-existing
```

注意：condition image 被 DQS、APS、KPS、MPS 共用。RQS/MCS/DCS 主要依赖生成资产的 rendered views 或 mask。

## 4. 七类评测指标

### 4.1 RQS - Render Quality Score

评价什么：

- 生成资产渲染图是否清晰。
- 轮廓、边缘、纹理、局部细节是否可辨。
- 是否有噪声、碎片、漂浮片、拉伸、异常渲染 artifact。

输入证据：

```text
benchmark/benchmark_assets/rendered_views/description/<source_folder>/<object_id>/000.png ... 029.png
benchmark/assets/quality_reference/image.png
```

代码会从 rendered views 中采样 4 张图给 VLM。prompt 让 VLM 输出 1 到 5 分，聚合器把它映射到 0 到 100。

### 4.2 MCS - Multi-view Consistency Score

评价什么：

- 多视角是否像同一个稳定的 3D 对象。
- 是否某些视角暴露坍缩、破洞、融化、异常缺失、异常突起。
- 材质和颜色是否跨视角一致。

输入证据：

```text
sampled rendered views
```

MCS 不评价类别是否正确，也不评价 affordance 或关节功能。它只问多视角是否能组成一个全局一致的对象。

### 4.3 DCS - Description Consistency Score

评价什么：

- 生成结果中的某个 mask 区域是否语义匹配 reference description。
- mask 是否精准落在被描述的 part 上，而不是覆盖全物体或大量无关区域。

输入证据：

```text
one color render
same-view black/white part mask
reference_description from descript_<dataset>.json
```

prompt 公式：

```text
DCS = round(0.60 * alignment_score + 0.40 * precision_score, 2)
```

其中：

- `alignment_score` 看语义是否对。
- `precision_score` 看 mask 是否定位准确。

### 4.4 DQS - Dimension Quality Score

评价什么：

- 生成资产的最大真实尺寸是否合理。

输入证据：

```text
condition image
basic_info.json / basic_info.txt 中的 dimension
或 physxgen 的 scale.npy
```

DQS 是两段式：

1. VLM 只看 condition image，估计真实世界尺寸 prior。
2. `score_dimension_results.py` 把方法输出的尺寸和 VLM 估计作对比。

代码公式：

```text
symmetric_error = 2 * abs(generated_max - estimated_max) / (generated_max + estimated_max)
if symmetric_error >= 0.8:
    DQS = 0
else:
    DQS = 100 * (1 - symmetric_error / 0.8)
```

### 4.5 APS - Affordance Plausibility Score

评价什么：

- affordance heatmap 的相对强弱排序是否符合人类交互常识。
- 关键交互区域是否高亮。
- 危险区域或不应交互区域是否被错误高亮。

输入证据：

```text
condition image
affordance heatmap views
```

APS 不看 heatmap 美观程度，而看“红色区域代表高 affordance”这一排序是否合理。

### 4.6 KPS - Kinematic Plausibility Score

评价什么：

- 可动部件是否符合图像先验。
- 新显露的隐藏结构或内部物体是否合理。
- 整体 articulation 是否连贯。

输入证据：

```text
condition image
standardized articulation video
image-prior JSON
```

KPS prompt 内部叫 VAPS，三部分权重：

```text
prior channel = 0.70
reveal channel = 0.20
global channel = 0.10
```

代码使用统一渲染器：

```text
benchmark/code/render/kinematic/kinematic_articulation_demo.py
```

不同方法的输入文件：

| 方法 | KPS 证据来源 |
|---|---|
| ours / physxanything | `basic.xml` |
| physxgen | `mesh/basic.urdf` |
| articulateanything | `joint_actor/iter_0/seed_0/mobility.urdf` |

### 4.7 MPS - Material Plausibility Score

评价什么：

- Young's modulus 对应的刚度、形变、回弹是否合理。
- Poisson's ratio 对应的横向膨胀和体积保持倾向是否合理。
- Density 对应的浮沉、惯性、碰撞质量感是否合理。

输入证据：

```text
condition image
water simulation video
floor simulation video
generated material parameters
```

MPS prompt 公式：

```text
weighted_score = 0.4 * S_E + 0.2 * S_nu + 0.4 * S_rho
MPS = 25 * (weighted_score - 1)
```

其中：

- `S_E` 是 Young's modulus 子分。
- `S_nu` 是 Poisson's ratio 子分。
- `S_rho` 是 density 子分。

## 5. 论文概念与当前开源代码的对应边界

论文里 PhysX-Bench 的 geometry 包含：

- CLIP score。
- 3D consistency。
- visual quality。

当前本地开源 benchmark 脚本中直接暴露的是：

- `RQS`：对应 visual quality。
- `MCS`：对应 3D consistency。
- 未看到同名 CLIP 评分脚本。

因此复现时要把两件事分开：

- 论文表格里的 Geometry 是论文报告层面的指标组。
- 当前公开代码主要提供 VLM-based RQS/MCS 和物理属性打分链路；CLIP 若要复现，需要另行确认作者是否在其它脚本、早期内部脚本或后续发布中提供。

## 6. 数据字段总览

Bench 的核心字段不是一个单一 schema，而是多层 schema：

| 层 | 核心字段 |
|---|---|
| HF 原始数据 | `demo_<dataset>/<object_id>.png`，`descript_<dataset>.json` |
| condition image | `dataset`，`object_id`，`first_frame.png` |
| method output | `source_folder`，`object_id`，`basic_info.json`，`basic.xml`，`basic.urdf`，`affordance/`，`description/` |
| metric manifest | `metric`，`method`，`dataset`，`object_id`，证据路径，`ready`，`status`，missing-zero 标记 |
| raw VLM result | `run_id`，`video_id`，`benchmark_context`，`turns_template`，`results[]`，`pair_error` |
| object score | `method`，`dataset`，`object_id`，`metric`，`score`，submetrics，`result_json` |
| summary score | `method`，`dataset`，`metric`，`count`，`mean`，`std` |
| denominator validation | `expected_count`，`raw_dedup_count`，`object_score_count`，`summary_count`，`count_mismatch` |

更详细字段字典见：

`C:\Users\robot\Documents\成长学习库\physx_omni_step5_bench_data_fields.md`

## 7. 为什么 denominator validation 是核心质量门

PhysX-Bench 的难点不是只打分，而是保证不同方法的分母一致。

如果某个方法缺少难样本的 rendered views、mask、heatmap、video，而聚合时直接跳过这些样本，它的均值会被虚高。因此代码设计了 missing-zero 和 denominator validation：

- missing RQS/MCS rendered views -> 0 分。
- missing DCS color/mask/description evidence -> DCS 0。
- malformed or missing DQS dimension -> DQS 0。
- missing APS heatmaps -> APS 0。
- missing KPS videos -> KPS 0。
- missing MPS water/floor video pairs -> MPS 0。

最终报告之前必须检查：

```text
count_mismatch == False
summary_count == expected_count
```

## 8. 和我们当前复现状态的关系

已经完成：

- 官方 demo 生成主流程跑通。
- benchmark tiny smoke 跑通 manifest、aggregation、denominator validation。
- 第五步已经把 Bench 的评测机制、数据字段和复现质量门整理成文档。

尚未完成：

- 全量准备 `physx_result/` 下所有方法输出。
- 全量生成 RQS/MCS/DCS/DQS/APS/KPS/MPS 证据资产。
- 全量运行 VLM 评分。
- 全量 denominator validation 后复现论文 PhysX-Bench 表格。

这意味着：当前可以讲清楚 PhysX-Bench 怎么设计、怎么跑、字段是什么；但还不能声称已经完整复现 PhysX-Bench 全表。


# PhysX-Omni 第五步附录：PhysX-Bench 数据字段字典

## 1. HF 原始数据层

官方数据集：

[https://huggingface.co/datasets/PhysX-Omni/PhysX-Bench](https://huggingface.co/datasets/PhysX-Omni/PhysX-Bench)

文件树：

```text
demo_inthewild/<object_id>.png
demo_mobility/<object_id>.png
demo_verse/<object_id>.png
descript_inthewild.json
descript_mobility.json
descript_verse.json
```

核心字段：

| 字段 | 类型 | 含义 |
|---|---|---|
| `dataset` | enum | `inthewild`、`mobility`、`verse` |
| `object_id` | string | 图片文件名去掉 `.png` 后的 ID |
| `image` | png | 条件图 |
| `reference_description` | string | 从 `descript_<dataset>.json` 按 `object_id` 读取 |

三个 description JSON 都是简单 key-value：

```json
{
  "object_id": "reference description"
}
```

## 2. Prepared condition image 层

转换脚本：

`benchmark/code/assets/prepare_demo_condition_images.py`

输入：

```text
physx_result/demo_<dataset>/<object_id>.png
```

输出：

```text
benchmark/benchmark_assets/condition_images/<dataset>/<object_id>/first_frame.png
```

字段：

| 字段 | 来源 | 用途 |
|---|---|---|
| `dataset` | 命令行参数 | 区分 mobility / verse / inthewild |
| `object_id` | png stem | 对齐 method output、description、manifest |
| `first_frame.png` | copy 或 symlink | DQS/APS/KPS/MPS 共用条件图 |

## 3. Method output 层

默认输出根目录：

```text
physx_result/<source_folder>/<object_id>/
```

`benchmark/scripts/common.sh` 中的映射：

| method | source folder |
|---|---|
| `ours` | `ours_<dataset>_181500` |
| `physxanything` | `output_physxanything_<dataset>` |
| `physxgen` | `outputs_physxgen_<dataset>` |
| `articulateanything` | `output_articulateanything_<dataset>` |

常见文件：

| 文件或目录 | 使用指标 | 含义 |
|---|---|---|
| `basic_info.json` / `basic_info.txt` | DQS / MPS | 对象名称、类别、尺寸、材料参数等 |
| `basic.xml` | KPS | ours / physxanything 的 MJCF 运动学证据 |
| `basic.urdf` | KPS 或导出检查 | URDF 表达 |
| `mesh/basic.urdf` | KPS | physxgen 的 URDF 输入 |
| `scale.npy` | DQS | physxgen 的尺度输出 |
| `affordance/*.png` | APS | affordance 灰度或 heatmap 源图 |
| `description/*.png` | DCS | part-level description mask |
| rendered mesh / glb | RQS / MCS / DCS | 多视角渲染源 |

## 4. Metric asset 层

| 资产 | 路径 | 使用指标 |
|---|---|---|
| condition images | `benchmark/benchmark_assets/condition_images/<dataset>/<object_id>/first_frame.png` | DQS / APS / KPS / MPS |
| rendered views | `benchmark/benchmark_assets/rendered_views/description/<source_folder>/<object_id>/<view>.png` | RQS / MCS / DCS |
| affordance heatmaps | `benchmark/benchmark_assets/affordance_heatmaps/<method>/<dataset>/<object_id>/` | APS |
| kinematic videos | `benchmark/benchmark_assets/kinematic_videos/<method>/<dataset>/<object_id>/kinematic_demo.mp4` | KPS |
| material videos | `benchmark/benchmark_assets/material_videos/...` and `material_videos_v2/floor/...` | MPS |
| watertight proxy meshes | `physx_result/watertightFix_max3000/<source_folder>/<object_id>/` | MPS preprocessing |

## 5. Manifest 公共字段

所有 manifest 都是 JSONL 和 CSV 双格式。最核心公共字段：

| 字段 | 含义 |
|---|---|
| `metric` | 指标名，如 `rqs`、`mcs`、`dcs`、`dqs`、`aps`、`kps`、`mps` |
| `method` | 方法名，如 `ours`、`physxanything`、`physxgen`、`articulateanything` |
| `dataset` | 数据域 |
| `object_id` | 样本 ID |
| `sample_id` | 通常等于 `object_id` |
| `relative_dir` | raw result 输出相对目录 |
| `source_result_dir` | 方法输出 object 目录 |
| `ready` | 该行是否有足够证据进入 VLM 或 deterministic scoring |
| `status` | 缺失或异常状态 |

## 6. RQS / MCS manifest 字段

构建脚本：

`benchmark/code/manifests/build_render_view_manifest.py`

字段：

| 字段 | 含义 |
|---|---|
| `source_folder` | 例如 `ours_mobility_181500` |
| `num_render_views_available` | 实际找到的 png 数 |
| `num_render_views_required` | RQS 需要 4，MCS 需要 8 |
| `num_render_views_selected` | 采样后送入 VLM 的数量 |
| `view_image_paths` | 被选择的 rendered views |
| `render_missing_score_zero` | 缺 rendered views 时是否自动 0 分 |

## 7. DCS manifest 字段

构建脚本：

`benchmark/code/manifests/build_description_mask_manifest.py`

字段：

| 字段 | 含义 |
|---|---|
| `part_id` | description/mask 对应 part |
| `render_folder` | rendered views 来源目录 |
| `result_folder` | 方法输出目录名 |
| `color_render_dir` | 彩色图目录 |
| `mask_dir` | description mask 目录 |
| `description_json` | `descript_<dataset>.json` |
| `reference_description` | 当前 object/part 的参考描述 |
| `num_color_views_available` | 彩色 views 数 |
| `num_description_mask_views_available` | mask views 数 |
| `num_paired_views_available` | 同视角配对数 |
| `num_nonblack_masks_available` | 非全黑 mask 数 |
| `view_pair_indices` | 选中的配对视角 |
| `selected_mask_white_ratio` | 选中 mask 的白色区域比例 |
| `view_image_paths` | 彩色图路径 |
| `mask_image_paths` | mask 图路径 |
| `dcs_missing_score_zero` | 证据缺失时是否 0 分 |

常见 `status`：

- `missing_color_render_views`
- `missing_description_masks`
- `missing_paired_description_views`
- `all_masks_black`
- `missing_reference_description`

## 8. DQS manifest 字段

构建脚本：

`benchmark/code/manifests/build_dimension_manifest.py`

字段：

| 字段 | 含义 |
|---|---|
| `image_path` / `condition_image` | 条件图 |
| `algorithm_dimension_source` | `basic_info_json`、`basic_info_txt`、`scale_npy`、`default_zero` 等 |
| `algorithm_json_path` | `basic_info.json` 路径 |
| `algorithm_info_path` | 实际采用的 info 文件 |
| `algorithm_scale_path` | `scale.npy` 路径 |
| `algorithm_dimension` | 原始 dimension 字符串，如 `180*120*150` |
| `algorithm_generated_max_dimension_cm` | 生成结果最大尺寸 |
| `algorithm_dimension_defaulted` | 是否因缺失或异常回退到 0 |
| `object_name` | 生成结果对象名 |
| `category` | 生成结果类别 |

DQS 特别点：

- VLM prior 行可以用 `--unique-priors` 生成，每个 dataset/object 只估一次尺寸。
- 真正 DQS 由 `score_dimension_results.py` 确定性计算。

## 9. APS manifest 字段

构建脚本：

`benchmark/code/manifests/build_affordance_manifest.py`

字段：

| 字段 | 含义 |
|---|---|
| `image_path` / `condition_image` | 条件图 |
| `affordance_heatmap_grid` | heatmap 拼图 |
| `affordance_view_paths` | 送入 VLM 的 heatmap views |
| `source_affordance_dir` | 原始 affordance 目录 |
| `num_affordance_views` | 采样后 views 数 |
| `num_affordance_views_available` | 原始可用 views 数 |
| `affordance_view_sample_indices` | 采样下标 |
| `affordance_view_sampling` | 当前为 `uniform_numeric` |
| `aps_missing_score_zero` | 是否因缺 heatmap 自动 0 |
| `missing_affordance_reason` | 缺失原因 |

常见 `status`：

- `missing_condition_image`
- `missing_affordance_heatmap_views`
- `insufficient_affordance_heatmap_views`
- `missing_affordance_heatmap_grid`

## 10. KPS manifest 字段

构建脚本：

`benchmark/code/manifests/build_kinematic_manifest.py`

字段：

| 字段 | 含义 |
|---|---|
| `image_path` | 条件图 |
| `video_path` | 实际采用的 articulation video |
| `expected_video_path` | 标准期望视频路径 |
| `fallback_video_path` | 兼容旧路径 |
| `video_source` | `expected` / `fallback` 等 |
| `source_xml` | `basic.xml` 路径 |
| `source_urdf` | URDF 路径 |

常见 `status`：

- `missing_condition_image`
- `missing_source_urdf`
- `needs_render_video`
- `missing_video_path`
- `uses_fallback_video`

## 11. MPS manifest 字段

构建脚本：

`benchmark/code/manifests/build_material_manifest.py`

字段：

| 字段 | 含义 |
|---|---|
| `image_path` | 条件图 |
| `video_paths` | water/floor 视频列表 |
| `floor_video_path` | floor simulation video |
| `water_video_path` | water simulation video |
| `material_parameters_path` | 材料参数 JSON |
| `source_folder` | 方法结果目录名 |
| `floor_folder` | floor video 所在目录 |
| `num_material_videos` | 实际可用视频数 |
| `num_material_videos_required` | 通常需要 2 |
| `mps_missing_score_zero` | 是否因缺材料视频/参数自动 0 |
| `missing_material_reason` | 缺失原因 |

常见 `status`：

- `missing_condition_image`
- `missing_floor_video`
- `missing_water_video`
- `missing_material_json`
- `invalid_floor_video`
- `invalid_water_video`
- `invalid_material_json`
- `insufficient_material_videos`

## 12. Raw VLM result 字段

生成脚本：

`benchmark/code/vlm/multi.py`

每个 object/metric 会输出：

```text
benchmark/benchmark_results/raw_vlm_outputs/<model_id>/run_<timestamp>/<relative_dir>/result.json
```

核心字段：

| 字段 | 含义 |
|---|---|
| `run_id` | 本次 VLM run |
| `video_path` | 单视频路径，KPS 等使用 |
| `video_paths` | 多视频路径，MPS 使用 |
| `video_id` | object id 或 sample id |
| `video_relative_dir` | 与 manifest `relative_dir` 对齐 |
| `paired_image_path` | 条件图或配对图 |
| `view_image_paths` | RQS/MCS/DCS views |
| `mask_image_paths` | DCS masks |
| `affordance_view_paths` | APS heatmap views |
| `benchmark_context` | 原始 manifest row |
| `sampling` | video/image 采样信息 |
| `turns_template` | 使用的 prompt turns |
| `results[]` | 每个 turn 的 VLM 输出 |
| `pair_error` | object-level 异常 |
| `elapsed_sec` | 该 pair 耗时 |

`results[]` 子字段：

| 字段 | 含义 |
|---|---|
| `turn_id` | prompt id，如 `render_quality`、`description_scoring` |
| `turn_index` | 第几个 turn |
| `prompt_ref_id` | prompt 引用 |
| `input_modalities` | 本 turn 使用 image/video/static/view 等输入 |
| `output` | VLM 输出的 JSON 字符串 |
| `error` | turn-level 异常 |
| `elapsed_sec` | turn 耗时 |

## 13. Aggregation 输出字段

聚合脚本：

`benchmark/code/aggregation/aggregate_vlm_results.py`

### object_scores.csv

字段：

```text
method,dataset,object_id,metric,score,
S_prior,S_reveal,S_global,
alignment_score,precision_score,
youngs_modulus_score,poisson_ratio_score,density_score,
task,verdict,turn_id,result_json,video_path,video_paths,paired_image_path,pair_error
```

解释：

- `S_prior/S_reveal/S_global` 主要来自 KPS/VAPS。
- `alignment_score/precision_score` 主要来自 DCS。
- `youngs_modulus_score/poisson_ratio_score/density_score` 主要来自 MPS。

### summary.csv

字段：

```text
method,dataset,metric,count,mean,std
```

这是报告 dataset-level benchmark 表格最直接的来源。

### submetric_summary.csv

用于展开 KPS/DCS/MPS 的子指标均值、标准差和计数。

## 14. Denominator validation 字段

校验脚本：

`benchmark/code/validation/validate_denominators.py`

字段：

| 字段 | 含义 |
|---|---|
| `method` | 方法 |
| `dataset` | 数据域 |
| `metric` | 指标 |
| `expected_count` | manifest 期望样本数 |
| `manifest_ready_count` | ready 样本数 |
| `manifest_missing_zero_count` | manifest 中应自动 0 分样本数 |
| `raw_metric_rows_before_dedup` | raw result 解析到的原始行数 |
| `raw_dedup_count` | 去重后的 raw 行数 |
| `raw_auto_zero_count` | 自动 0 分 raw 行数 |
| `object_score_count` | object-level 分数行数 |
| `summary_count` | summary 中 count |
| `summary_mean` | summary 均值 |
| `duplicate_object_result_keys` | 重复 object 结果数 |
| `missing_raw_after_dedup` | manifest 有但 raw 缺失数 |
| `extra_raw_after_dedup` | raw 多出来的 object 数 |
| `count_mismatch` | 是否分母不一致 |

报告任何 benchmark 分数前，`count_mismatch` 必须为 `False`。


# Notebook checks

??????????????????????????Hugging Face ???????? smoke ???????????????????????????????


In [1]:
from pathlib import Path
import json, ast, re
import pandas as pd

workspace = Path.cwd()
repo = workspace / "physx-omni-assets" / "code" / "PhysX-Omni"
bench = repo / "benchmark"
print("repo", repo, repo.exists())
print("benchmark", bench.exists())


repo C:\Users\robot\Documents\成长学习库\physx-omni-assets\code\PhysX-Omni True
benchmark True


In [2]:
# Hugging Face PhysX-Bench file tree count.
from huggingface_hub import HfApi
api = HfApi()
files = api.list_repo_files(repo_id="PhysX-Omni/PhysX-Bench", repo_type="dataset")
counts = {}
for p in files:
    top = p.split('/')[0]
    counts[top] = counts.get(top, 0) + 1
pd.DataFrame(sorted(counts.items()), columns=["top_level", "file_count"])


,top_level,file_count
0,.gitattributes,1
1,README.md,1
2,demo_inthewild,426
3,demo_mobility,388
4,demo_verse,400
5,descript_inthewild.json,1
6,descript_mobility.json,1
7,descript_verse.json,1


In [3]:
# Local description JSON counts and basic text length stats.
desc_root = bench / "assets" / "description"
rows = []
for path in sorted(desc_root.glob("descript_*.json")):
    data = json.loads(path.read_text(encoding="utf-8"))
    lengths = [len(v) for v in data.values()]
    rows.append({
        "file": path.name,
        "count": len(data),
        "min_chars": min(lengths),
        "max_chars": max(lengths),
        "avg_chars": round(sum(lengths)/len(lengths), 1),
        "sample_key": next(iter(data.keys())),
        "sample_description": next(iter(data.values())),
    })
pd.DataFrame(rows)


,file,count,min_chars,max_chars,avg_chars,sample_key,sample_description
0,descript_inthewild.json,426,21,97,43.5,0,A heart-shaped diamond made of carbon.
1,descript_mobility.json,388,21,157,98.8,103189,This is the left leg (temple arm) of the eyegl...
2,descript_verse.json,400,16,175,64.1,00d1cb5aa82745228a3b764c97f867de,The temples are the long arms extending from t...


In [4]:
# Manifest CSV fieldnames extracted from builder scripts.
manifest_root = bench / "code" / "manifests"
rows = []
for path in sorted(manifest_root.glob("build_*_manifest.py")):
    text = path.read_text(encoding="utf-8")
    m = re.search(r"fieldnames\s*=\s*\[(.*?)\]", text, re.S)
    if not m:
        continue
    vals = ast.literal_eval("[" + m.group(1) + "]")
    rows.append({"builder": path.name, "field_count": len(vals), "fields": ", ".join(vals)})
pd.DataFrame(rows)


,builder,field_count,fields
0,build_affordance_manifest.py,18,"metric, method, dataset, object_id, image_path..."
1,build_description_mask_manifest.py,27,"metric, method, dataset, object_id, part_id, r..."
2,build_dimension_manifest.py,18,"metric, method, dataset, object_id, image_path..."
3,build_kinematic_manifest.py,15,"metric, method, dataset, object_id, image_path..."
4,build_material_manifest.py,18,"metric, method, dataset, object_id, image_path..."
5,build_render_view_manifest.py,15,"metric, method, dataset, object_id, sample_id,..."


In [5]:
# Status tokens used by manifest builders.
status_rows=[]
for path in sorted((bench / "code" / "manifests").glob("build_*_manifest.py")):
    text = path.read_text(encoding="utf-8")
    statuses = sorted(set(re.findall(r'status\.append\("([^"]+)"\)', text)))
    if statuses:
        status_rows.append({"builder": path.name, "statuses": ", ".join(statuses)})
pd.DataFrame(status_rows)


,builder,statuses
0,build_affordance_manifest.py,"insufficient_affordance_heatmap_views, missing..."
1,build_description_mask_manifest.py,"all_masks_black, missing_color_render_views, m..."
2,build_dimension_manifest.py,missing_condition_image
3,build_kinematic_manifest.py,"missing_condition_image, missing_source_urdf, ..."
4,build_material_manifest.py,"insufficient_material_videos, invalid_floor_vi..."


In [6]:
# Aggregation output schemas.
text = (bench / "code" / "aggregation" / "aggregate_vlm_results.py").read_text(encoding="utf-8")
object_fields = ast.literal_eval("[" + re.search(r"object_fields\s*=\s*\[(.*?)\]", text, re.S).group(1) + "]")
submetric_fields = ast.literal_eval("[" + re.search(r"submetric_fields\s*=\s*\[(.*?)\]", text, re.S).group(1) + "]")
print("object_fields", len(object_fields), object_fields)
print("summary_fields", ['method','dataset','metric','count','mean','std'])
print("submetric_fields", len(submetric_fields), submetric_fields)


object_fields 21 ['method', 'dataset', 'object_id', 'metric', 'score', 'S_prior', 'S_reveal', 'S_global', 'alignment_score', 'precision_score', 'youngs_modulus_score', 'poisson_ratio_score', 'density_score', 'task', 'verdict', 'turn_id', 'result_json', 'video_path', 'video_paths', 'paired_image_path', 'pair_error']
summary_fields ['method', 'dataset', 'metric', 'count', 'mean', 'std']
submetric_fields 30 ['method', 'dataset', 'metric', 'count', 'score_mean', 'score_std', 'S_prior_mean', 'S_prior_std', 'S_prior_count', 'S_reveal_mean', 'S_reveal_std', 'S_reveal_count', 'S_global_mean', 'S_global_std', 'S_global_count', 'alignment_score_mean', 'alignment_score_std', 'alignment_score_count', 'precision_score_mean', 'precision_score_std', 'precision_score_count', 'youngs_modulus_score_mean', 'youngs_modulus_score_std', 'youngs_modulus_score_count', 'poisson_ratio_score_mean', 'poisson_ratio_score_std', 'poisson_ratio_score_count', 'density_score_mean', 'density_score_std', 'density_score_c

In [7]:
# Already generated tiny smoke denominator validation.
smoke = bench / "tiny_example" / "generated" / "benchmark_results"
for name in ["summary.csv", "denominator_validation.csv", "object_scores.csv"]:
    path = smoke / name
    print(path, path.exists())
    if path.exists():
        display(pd.read_csv(path))


C:\Users\robot\Documents\成长学习库\physx-omni-assets\code\PhysX-Omni\benchmark\tiny_example\generated\benchmark_results\summary.csv True


,method,dataset,metric,count,mean,std
0,ours,mobility,RQS,1,100.0,0.0


C:\Users\robot\Documents\成长学习库\physx-omni-assets\code\PhysX-Omni\benchmark\tiny_example\generated\benchmark_results\denominator_validation.csv True


,method,dataset,metric,expected_count,manifest_ready_count,manifest_missing_zero_count,raw_metric_rows_before_dedup,raw_dedup_count,raw_auto_zero_count,object_score_count,summary_count,summary_mean,duplicate_object_result_keys,missing_raw_after_dedup,extra_raw_after_dedup,missing_object_ids_sample,extra_raw_object_ids_sample,count_mismatch
0,ours,mobility,RQS,1,1,0,1,1,0,1,1,100.0,0,0,0,NaN,NaN,False


C:\Users\robot\Documents\成长学习库\physx-omni-assets\code\PhysX-Omni\benchmark\tiny_example\generated\benchmark_results\object_scores.csv True


,method,dataset,object_id,metric,score,S_prior,S_reveal,S_global,alignment_score,precision_score,...,poisson_ratio_score,density_score,task,verdict,turn_id,result_json,video_path,video_paths,paired_image_path,pair_error
0,ours,mobility,1,RQS,100.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,render_quality,benchmark\tiny_example\generated\benchmark_res...,NaN,[],NaN,NaN
